In [1]:
import os
import sys
import pickle as pkl
import pandas as pd
import numpy as np
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

/home/cyrus/venvs/uts-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import Dataset
Dataset contains SBERT question, human response, agent response, and agent summary vectors. 

In [2]:
sys.path.append(str(Path.cwd().parent))

ROOT    = Path(os.getcwd()).resolve().parents[0]
DATA    = ROOT / "data"
WEIGHTS = ROOT / "weights"

with open(DATA / "concat_train_set.pkl", "rb") as file:
    train_set = pkl.load(file)

with open(DATA / "concat_test_set.pkl", "rb") as file:
    test_set = pkl.load(file)

train_df    = pd.DataFrame(train_set)
test_df     = pd.DataFrame(test_set)

print("Train set:", train_df.shape)
print("Test set:", train_df.shape)

Train set: (1708, 12)
Test set: (1708, 12)


## Load BERT Models

In [3]:
import torch
from dataclasses import dataclass, field
from transformers import (
    BertForQuestionAnswering, 
    AutoTokenizer,
)
from sentence_transformers import SentenceTransformer

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [5]:
bert_model = BertForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")
bert_model.to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2').to(device)

# Count BERT's total number of trainable parameters by summing the size of every weight tensor.
bert_num_params = sum(p.numel() for p in bert_model.parameters())
# Print the count, so "small model" isn't just a label in a markdown cell -- see the real number.
print(f"BERT ({bert_model.config.name_or_path}) parameters: {bert_num_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18469.75it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18791.36it/s]


BERT (deepset/bert-base-cased-squad2) parameters: 107,721,218


In [6]:
@dataclass
class Config:
    # Training config
    epochs: int         = 50

    # Data shape
    question_dim: int   = sentence_model.get_sentence_embedding_dimension()
    bert_dim: int       = sentence_model.get_sentence_embedding_dimension()

    # Latent sizes
    style_dim: int      = sentence_model.get_sentence_embedding_dimension()
    content_dim: int    = sentence_model.get_sentence_embedding_dimension()
    context_dim: int    = sentence_model.get_sentence_embedding_dimension()

    disc_dim: int       = sentence_model.get_sentence_embedding_dimension()
    gene_dim: int       = sentence_model.get_sentence_embedding_dimension()

    # Optimisation
    lr_generator: float     = 1e-4
    lr_discriminator: float = 1e-4
    batch_size: int         = 32

cfg = Config(epochs=50)

/tmp/ipykernel_91849/691886234.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  question_dim: int   = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91849/691886234.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bert_dim: int       = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91849/691886234.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  style_dim: int      = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91849/691886234.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  content_dim: int    = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91849/691886234.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `ge

In [7]:
alt_state = torch.load(WEIGHTS / "alt_state.pt", weights_only=False, map_location="cpu")
mix_state = torch.load(WEIGHTS / "mix_state.pt", weights_only=False, map_location="cpu")

alt_config = alt_state["config"]
mix_config = mix_state["config"]

alt_model = alt_state["model"]
mix_model = mix_state["model"]

alt_model.to(device)
mix_model.to(device)

SBERTDisentangler(
  (style_encoder): StyleEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=16, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=16, out_features=16, bias=True)
    )
  )
  (content_encoder): ContentEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (context_encoder): ContextEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (generator): SBERTGenerator(
    (mlp): Sequential(
      (0): Linear(in_features=272, out_features=384, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (discriminator): SBERTDiscriminator(
    (mlp): Sequenti

## Probe Training Configuration

In [8]:
@torch.no_grad()
def compute_baselines(model, loader, device):
    model.eval()

    all_questions = []
    all_human_style = []
    all_agent_style = []
    all_human_content = []
    all_agent_content = []
    all_human_context = []
    all_agent_context = []

    for batch in loader:
        question = batch["question"].to(device)
        human = batch["human"].to(device)
        agent = batch["agent"].to(device)

        # Encode the paired human and AI responses
        style_h, content_h, context_h = model.encode(human)
        style_a, content_a, context_a = model.encode(agent)

        all_questions.append(question.cpu())
        all_human_style.append(style_h.cpu())
        all_agent_style.append(style_a.cpu())
        all_human_content.append(content_h.cpu())
        all_agent_content.append(content_a.cpu())
        all_human_context.append(context_h.cpu())
        all_agent_context.append(context_a.cpu())

    # Concatenate all examples across batches
    human_style = torch.cat(all_human_style, dim=0)
    agent_style = torch.cat(all_agent_style, dim=0)
    human_content = torch.cat(all_human_content, dim=0)
    agent_content = torch.cat(all_agent_content, dim=0)
    human_context = torch.cat(all_human_context, dim=0)
    agent_context = torch.cat(all_agent_context, dim=0)

    # Pool human and agent targets for each representation.
    # Each response receives equal weight.
    all_content = torch.cat([human_content, agent_content], dim=0)
    all_context = torch.cat([human_context, agent_context], dim=0)

    # Calculate the mean target vectors
    baselines = {
        "content": all_content.mean(dim=0),
        "context": all_context.mean(dim=0),
    }

    return baselines

In [9]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=True) 

/tmp/ipykernel_91849/3805931428.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)


## Evaluate Probes

In [10]:
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
probe_specs = {
    "style_content":   ("style", "content"),
    "style_context":   ("style", "context"),
    "content_style":   ("content", "style"),
    "content_context": ("content", "context"),
    "context_style":   ("context", "style"),
    "context_content": ("context", "content"),
}

model_configs = [
    (mix_model, "mix"),
    (alt_model, "alt"),
]

In [12]:
@torch.no_grad()
def extract_representations(model, loader, device):
    model.eval()

    codes = {
        split: {name: [] for name in ("style", "content", "context")}
        for split in ("human", "agent")
    }

    for batch in loader:
        human = batch["human"].to(device)
        agent = batch["agent"].to(device)

        h_s, h_c, h_x = model.encode(human)
        a_s, a_c, a_x = model.encode(agent)

        for name, h, a in (
            ("style", h_s, a_s),
            ("content", h_c, a_c),
            ("context", h_x, a_x),
        ):
            codes["human"][name].append(h.cpu().numpy())
            codes["agent"][name].append(a.cpu().numpy())

    # Keep every representation aligned with its source label.
    result = {}

    for split in ("human", "agent"):
        result[split] = {
            name: np.concatenate(values, axis=0)
            for name, values in codes[split].items()
        }

    return result

def combine_responses(representations):
    """Concatenate human and agent responses in matching order."""
    X = {
        name: np.concatenate([
            representations["human"][name],
            representations["agent"][name],
        ], axis=0)
        for name in ("style", "content", "context")
    }

    y_source = np.concatenate([
        np.zeros(len(representations["human"]["style"]), dtype=int),
        np.ones(len(representations["agent"]["style"]), dtype=int),
    ])

    return X, y_source

In [13]:
results = {}

for model, model_name in model_configs:
    print(f"\n{'=' * 76}")
    print(f"{model_name.upper()} MODEL — TEST RESULTS")
    print(f"{'=' * 76}")

    train_repr = extract_representations(model, train_loader, device)
    test_repr = extract_representations(model, test_loader, device)

    X_train, y_train = combine_responses(train_repr)
    X_test, y_test = combine_responses(test_repr)

    # Mean-vector baselines are fitted using TRAIN data only.
    baselines = {
        target: X_train[target].mean(axis=0)
        for target in ("content", "context")
    }

    results[model_name] = {}

    for probe_name, (source, target) in probe_specs.items():
        Xtr = X_train[source]
        Xte = X_test[source]

        # Classification: predict whether the response is human or AI.
        if target == "style":
            probe = make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=2000)
            )

            probe.fit(Xtr, y_train)
            predictions = probe.predict(Xte)
            accuracy = accuracy_score(y_test, predictions)

            results[model_name][probe_name] = {"accuracy": accuracy}

            print(
                f"{probe_name:20s} | "
                f"Accuracy: {accuracy:.4f} | Chance: 0.5000"
            )

        # Regression: predict the target representation.
        else:
            Y_train = X_train[target]
            Y_test = X_test[target]

            probe = make_pipeline(
                StandardScaler(),
                Ridge(alpha=1.0)
            )

            probe.fit(Xtr, Y_train)
            Y_pred = probe.predict(Xte)

            # Per-example cosine similarity.
            cosine = np.diag(cosine_similarity(Y_pred, Y_test)).mean()
            mse = mean_squared_error(Y_test, Y_pred)

            # Mean-vector baseline, calculated from training targets.
            baseline_pred = np.broadcast_to(baselines[target], Y_test.shape)
            baseline_cosine = np.diag(cosine_similarity(baseline_pred, Y_test)).mean()
            baseline_mse = mean_squared_error(Y_test, baseline_pred)

            results[model_name][probe_name] = {
                "cosine": cosine,
                "mse": mse,
                "baseline_cosine": baseline_cosine,
                "baseline_mse": baseline_mse,
                "cosine_gain": cosine - baseline_cosine,
                "mse_reduction": baseline_mse - mse,
            }

            print(
                f"{probe_name:20s} | "
                f"Probe: Cos={cosine:.4f}, MSE={mse:.6f} | "
                f"Baseline: Cos={baseline_cosine:.4f}, "
                f"MSE={baseline_mse:.6f} | "
                f"ΔCos={cosine - baseline_cosine:+.4f}, "
                f"ΔMSE={baseline_mse - mse:+.6f}"
            )


MIX MODEL — TEST RESULTS
style_content        | Probe: Cos=0.6189, MSE=0.040435 | Baseline: Cos=0.4934, MSE=0.051772 | ΔCos=+0.1256, ΔMSE=+0.011337
style_context        | Probe: Cos=0.9930, MSE=0.001223 | Baseline: Cos=0.9918, MSE=0.002865 | ΔCos=+0.0013, ΔMSE=+0.001643
content_style        | Accuracy: 0.7421 | Chance: 0.5000
content_context      | Probe: Cos=0.9976, MSE=0.000520 | Baseline: Cos=0.9918, MSE=0.002865 | ΔCos=+0.0058, ΔMSE=+0.002345
context_style        | Accuracy: 0.7053 | Chance: 0.5000
context_content      | Probe: Cos=0.9494, MSE=0.006355 | Baseline: Cos=0.4934, MSE=0.051772 | ΔCos=+0.4560, ΔMSE=+0.045418

ALT MODEL — TEST RESULTS
style_content        | Probe: Cos=0.6074, MSE=0.023077 | Baseline: Cos=0.5289, MSE=0.027186 | ΔCos=+0.0785, ΔMSE=+0.004109
style_context        | Probe: Cos=0.9862, MSE=0.000890 | Baseline: Cos=0.9849, MSE=0.001052 | ΔCos=+0.0013, ΔMSE=+0.000162
content_style        | Accuracy: 0.6921 | Chance: 0.5000
content_context      | Probe: Cos=0.994